# PySpark drills: execution and production

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-drills-12-execution-and-production.ipynb)

Practice lessons 1 and 12–15 through small operational scenarios. Predict the outcome before running the solution.

Run the input cell, try the exercise, then reveal the solution. The handbook site collapses tagged solutions. Jupyter and Colab use the saved collapse metadata when supported.

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## Exercise 1: Identify a shuffle in a physical plan

**Input.** `partitioned` has IDs 0 through 11 across three partitions. Compare filtering to `id > 2` with grouping those survivors by parity.

**Task.** With AQE disabled for this inspection, compare a filter to a groupBy on a derived key. Report whether each plan exchanges rows. Explain what action would execute the plan.

In [ ]:
from pyspark.sql import functions as F
partitioned = spark.range(12, numPartitions=3)

**Expected output**

| operation | exchanges |
| --- | --- |
| filter | false |
| group | true |

In [ ]:
# Write your solution here.

In [ ]:
before = spark.conf.get("spark.sql.adaptive.enabled")
try:
    spark.conf.set("spark.sql.adaptive.enabled", "false")
    filtered = partitioned.where("id > 2")
    grouped = filtered.groupBy((F.col("id") % 2).alias("parity")).count()
    plans = [(name, "Exchange" in frame._jdf.queryExecution().executedPlan().toString())
             for name, frame in [("filter", filtered), ("group", grouped)]]
finally:
    spark.conf.set("spark.sql.adaptive.enabled", before)
result = spark.createDataFrame(plans, ["operation", "exchanges"]).orderBy("operation")
result.show()

## Exercise 2: Choose coalesce or repartition

**Input.** `partition_input` has 12 rows, with IDs 0 through 11, in four partitions.

**Task.** Start with four partitions. Show that coalesce can reduce to two without redistributing all rows, but cannot grow to six. Grow to six with repartition and preserve all twelve records.

In [ ]:
partition_input = spark.range(12, numPartitions=4)

**Expected output**

| operation | partitions | records |
| --- | --- | --- |
| coalesce_down | 2 | 12 |
| coalesce_up | 4 | 12 |
| repartition_up | 6 | 12 |

In [ ]:
# Write your solution here.

In [ ]:
variants = [
    ("coalesce_down", partition_input.coalesce(2)),
    ("coalesce_up", partition_input.coalesce(6)),
    ("repartition_up", partition_input.repartition(6)),
]
result = spark.createDataFrame(
    [(name, frame.rdd.getNumPartitions(), frame.count()) for name, frame in variants],
    ["operation", "partitions", "records"],
).orderBy("operation")
result.show()

## Exercise 3: Diagnose symptoms from stage metrics

**Input.** The input has one row per stage, with median and longest task time in seconds, and spill and driver collection in MB.

**Task.** Classify each stage using these rules in order. Driver collection above 1,000 MB is driver risk. A longest task above 5 times the median is skew. Otherwise any spill is memory pressure. Return the stage name and diagnosis in alphabetical stage order. As a follow-up, discuss which metric you would inspect next.

In [ ]:
metrics = spark.createDataFrame(
    [("collect", 3, 4, 0, 1500), ("hot_join", 4, 40, 200, 0), ("sort", 10, 12, 100, 0)],
    "stage string, median_s int, max_s int, spilled_mb int, collected_mb int",
)

**Expected output**

| stage | diagnosis |
| --- | --- |
| collect | driver_risk |
| hot_join | skew |
| sort | spill |

In [ ]:
# Write your solution here.

In [ ]:
diagnosis = (F.when(F.col("collected_mb") > 1000, "driver_risk")
             .when(F.col("max_s") > 5 * F.col("median_s"), "skew")
             .when(F.col("spilled_mb") > 0, "spill").otherwise("healthy"))
result = metrics.select("stage", diagnosis.alias("diagnosis")).orderBy("stage")
result.show()

## Exercise 4: Gate a release on the schema contract

**Input.** The contract requires integer id, decimal amount and a date. The candidate has integer id, string amount and a channel field.

**Task.** Compare an incoming schema against required field types. Report one missing field and one wrong type. Extra fields are allowed. Explain whether a retry of the same input can fix either failure.

In [ ]:
candidate = spark.createDataFrame([(1, "12.50", "web")], "id int, amount string, channel string")
contract = {"id": "int", "amount": "decimal(10,2)", "order_date": "date"}

**Expected output**

| field | issue |
| --- | --- |
| amount | wrong_type |
| order_date | missing |

In [ ]:
# Write your solution here.

In [ ]:
actual_schema = dict(candidate.dtypes)
violations = [(name, "missing" if name not in actual_schema else "wrong_type")
              for name, expected in contract.items() if actual_schema.get(name) != expected]
result = spark.createDataFrame(violations, "field string, issue string").orderBy("field")
result.show()

## Exercise 5: Preserve one result when salting a skewed join

**Input.** Four orders refer to two customers. Customer 0 appears on three orders, and customer 1 appears on one.

**Task.** Customer 0 is a hot key. Split its fact rows across two salts based on order ID parity and copy its dimension row to both salts. Keep all other keys on salt 0. Verify that the salted join returns the same rows as the ordinary join.

In [ ]:
facts = spark.createDataFrame([(1, 0), (2, 0), (3, 0), (4, 1)], "id int, customer int")
regions = spark.createDataFrame([(0, "west"), (1, "east")], "customer int, region string")

**Expected output**

| id | region |
| --- | --- |
| 1 | west |
| 2 | west |
| 3 | west |
| 4 | east |

In [ ]:
# Write your solution here.

In [ ]:
salted_facts = facts.withColumn("salt", F.when(F.col("customer") == 0, F.col("id") % 2).otherwise(0))
salted_regions = regions.withColumn("salt", F.explode(
    F.when(F.col("customer") == 0, F.array(F.lit(0), F.lit(1))).otherwise(F.array(F.lit(0)))))
result = salted_facts.join(salted_regions, ["customer", "salt"]).select("id", "region").orderBy("id")
baseline = facts.join(regions, "customer").select("id", "region")
assert result.exceptAll(baseline).count() == baseline.exceptAll(result).count() == 0
result.show()

## Exercise 6: Verify a Parquet filter reaches the scan

**Input.** Order 1 has an amount of 10 and order 2 has an amount of 30.

**Task.** Write two orders to Parquet, one above the amount threshold and one below. Read the file, filter for amounts above 20, and return only order IDs. Confirm that the formatted plan lists the amount comparison under PushedFilters.

In [ ]:
import tempfile
from contextlib import redirect_stdout
from io import StringIO

orders_for_scan = spark.createDataFrame([(1, 10), (2, 30)], "id int, amount int")

**Expected output**

| id |
| --- |
| 2 |

In [ ]:
# Write your solution here.

In [ ]:
with tempfile.TemporaryDirectory() as directory:
    path = f"{directory}/orders"
    orders_for_scan.write.parquet(path)
    filtered_scan = spark.read.parquet(path).where("amount > 20").select("id")
    plan = StringIO()
    with redirect_stdout(plan):
        filtered_scan.explain("formatted")
    assert any("PushedFilters" in line and "GreaterThan(amount,20)" in line
               for line in plan.getvalue().splitlines())
    matching_ids = filtered_scan.collect()
result = spark.createDataFrame(matching_ids, "id int").orderBy("id")
result.show()